# cmri_model | Locate original figures and review saved Jupyter results

**Read-only recovery notebook.** First search all known storage locations for the old 192-molecule, H200 10K, chemistry, duplication, and downstream figures; then display *the files that actually exist* and inspect their underlying CSV/JSON/NPY data. Nothing is deleted, overwritten, retrained, or inferred from missing files.

Use **Run → Run All Cells** in Jupyter once you have checked the roots below. This notebook is compatible with a CPU kernel (requires pandas, numpy, matplotlib, PIL, IPython). It does **not** load checkpoint weights or the 15M LMDB records.

**Important distinction:** `runs/evaluation/*_best_embeddings/full_n192_seed17/` is the expected output directory of the **earlier 192-molecule** notebooks, whereas `heart_evaluation/general_audit_10k_v2/` contains the later 10K evaluation. **Never relabel T4 results as H200.**

In [ ]:
from pathlib import Path
import os, re, json, csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

REPO = Path('/home/ypan/heart_liver/model_train/heart-molecular-pretraining')
EVAL = Path('/project/yli4/ypan/heart_evaluation/general_audit_10k_v2')
AUDITS = Path('/project/yli4/ypan/audits')
OUTPUT = REPO / 'summary_figures' / 'original_figure_recovery'
# Inventory only; make the output folder for its CSV summaries, do not change any source.
OUTPUT.mkdir(parents=True, exist_ok=True)
print('REPO exists:', REPO.exists())
print('EVAL exists:', EVAL.exists())
print('AUDITS exists:', AUDITS.exists())

## 1. Check exact destinations recorded in the original Jupyter notebooks

The T4/H200 `HEART_*checkpoint_best_embedding_evaluation.ipynb` source codes define the following `OUTPUT_DIR / "full_n192_seed17"`. These are candidates, not guaranteed output files.

In [ ]:
EXPECTED = {
    'T4 early n192': REPO/'runs/evaluation/t4_best_embeddings/full_n192_seed17',
    'H200 early n192': REPO/'runs/evaluation/h200_best_embeddings/full_n192_seed17',
    'H200 10K raw evaluation': EVAL/'checkpoint_evaluation/full_10k_batch8',
    'H200 10K existing PCA': EVAL/'checkpoint_evaluation/full_10k_batch8/figures',
    'H200 training plots': EVAL/'training_history',
    'Chemistry slide evidence': EVAL/'chemistry_slide_evidence',
    'Chemistry derived slides': EVAL/'chemistry_slide_evidence/slides_from_existing_10k',
    'Downstream benchmarks': EVAL/'chemistry_slide_evidence/independent_property_benchmarks',
    'Duplication audits': AUDITS,
    'New composed figures': REPO/'summary_figures',
}
rows=[]
for label,d in EXPECTED.items():
    direct = sorted(q.name for q in d.iterdir())[:15] if d.is_dir() else []
    rows.append({'location':label,'exists':d.is_dir(),'path':str(d), 'sample_contents':', '.join(direct)})
display(pd.DataFrame(rows))

## 2. Search figure filenames without traversing millions of LMDB shards

Search the most relevant *output* folders first, plus `runs/evaluation`. A bounded walk prunes `lmdb`, `checkpoints`, `site-packages`, `.git`, and other high-volume directories. The result includes the **exact file path**, so you can open it from Jupyter File Browser.

In [ ]:
FIG_EXTS={'.png','.svg','.pdf','.jpg','.jpeg','.webp'}
KEYWORDS=['pca','embedding','diagnostic','geometry','coord','masked','atom','accuracy','benchmark','esol','freesolv','sampl','lipophilicity','duplicate','dedup','scaffold','train_vs','validation','h200']
SCAN_ROOTS=[
    REPO/'runs/evaluation', REPO/'runs',
    EVAL, AUDITS, REPO/'summary_figures',
    REPO/'model', REPO/'model/pretrain/notebooks',
]
PRUNE={'.git','__pycache__','.conda','site-packages','node_modules','lmdb','lmdb_full','shards','checkpoints','cache','tmp'}
found={}
for root in SCAN_ROOTS:
    if not root.exists(): continue
    root=root.resolve()
    for folder, dirs, files in os.walk(root):
        depth=len(Path(folder).relative_to(root).parts)
        dirs[:]=[d for d in dirs if d not in PRUNE and depth<9]
        for filename in files:
            f=Path(folder)/filename
            if f.suffix.lower() in FIG_EXTS and any(k in f.name.lower() for k in KEYWORDS):
                found[str(f.resolve())]={'name':f.name,'path':str(f.resolve()), 'size_kb':round(f.stat().st_size/1024,1), 'source_root':str(root)}
figures=pd.DataFrame(found.values())
if len(figures):
    figures=figures.sort_values(['name','path']).reset_index(drop=True)
    display(figures)
    figures.to_csv(OUTPUT/'original_figures_found.csv',index=False)
    print('Found:',len(figures),'files. Inventory:',OUTPUT/'original_figures_found.csv')
else:
    print('No candidate images in these output roots. Expand SCAN_ROOTS if needed.')

## 3. Show original images where they still exist

The gallery displays **original image files**, not reconstructed or newly synthesized plots. Large previews are resized only in memory for display. Use `SHOW_KEYS` to target different filenames.

In [ ]:
SHOW_KEYS=['pca.png','pca_by_representation','embedding_diagnostics','atom_and_geometry','detailed_validation','atom_accuracy','pca_chemistry','benchmark_esol','benchmark_freesolv','benchmark_lipophilicity','duplicate']
if 'figures' in globals() and not figures.empty:
    matched=figures[figures['name'].str.lower().apply(lambda x:any(k in x for k in SHOW_KEYS))]
    # Avoid displaying hundreds of matches or PDF first pages; only PNG/JPG, first 18.
    matched=matched[matched['name'].str.lower().str.endswith(('.png','.jpg','.jpeg'))].head(18)
    for _,r in matched.iterrows():
        print('ORIGINAL FILE:',r['path'])
        try:
            with Image.open(r['path']) as im:
                im=im.convert('RGB')
                im.thumbnail((1150,760))
                display(im.copy())
        except Exception as exc: print('Could not preview:',exc)
else: print('No image inventory was found.')

## 3B. Complete visual inventory — every recovered original image

This is the **what do we still have?** step. Displays thumbnails for **all HPC images found** by the scanner, grouped into pages of 12 with full source paths recorded in the inventory. This gallery contains only files that actually exist on the HPC and is distinct from the bundled offline archive. The preview is displayed in Jupyter; source files are never modified.


In [ ]:
from math import ceil
from PIL import ImageDraw, ImageFont
from IPython.display import display

def contact_sheets_from_inventory(figures, page_size=12, image_size=(380,235)):
    if figures is None or figures.empty:
        print('No images found on this HPC by the earlier search. Check SCAN_ROOTS and exact expected paths.')
        return
    files = [Path(x) for x in figures['path'] if Path(x).suffix.lower() in {'.png','.jpg','.jpeg','.webp'} and Path(x).is_file()]
    print(f'HPC gallery: {len(files)} actual image files; output shown in pages of {page_size}.')
    for page_start in range(0,len(files),page_size):
        subset=files[page_start:page_start+page_size]
        cw,ch=image_size[0]+20,image_size[1]+52
        canvas=Image.new('RGB',(3*cw,4*ch),'#f8f6f2')
        draw=ImageDraw.Draw(canvas)
        for i,p in enumerate(subset):
            x=(i%3)*cw;y=(i//3)*ch
            draw.rectangle((x+3,y+3,x+cw-4,y+ch-4),fill='white',outline='#dad3c9',width=2)
            try:
                with Image.open(p) as im:
                    im=im.convert('RGB')
                    im.thumbnail(image_size)
                    canvas.paste(im,(x+10+(image_size[0]-im.width)//2,y+7+(image_size[1]-im.height)//2))
                draw.text((x+12,y+ch-42),p.name[:48],fill='#222222')
                draw.text((x+12,y+ch-25),str(p.parent)[-62:],fill='#666666')
            except Exception as e:
                draw.text((x+12,y+50),f'Cannot render: {type(e).__name__}',fill='#aa3344')
        display(canvas)
        print(f'Gallery page {page_start//page_size+1}/{ceil(len(files)/page_size)}; source paths are in original_figures_found.csv')

contact_sheets_from_inventory(figures if 'figures' in globals() else None)


## 3C. Preview bundled archive copies (if you extracted the full recovery ZIP)

If the original HPC copy is missing but you uploaded and unzipped the complete recovery package, the `archived_general_pretraining_figures_v3` folder contains 12 saved images. Those are **archival copies**, not proof of current HPC file existence. This cell previews them separately.


In [ ]:
archive_candidates=[REPO/'archived_general_pretraining_figures_v3', Path.cwd()/'archived_general_pretraining_figures_v3', Path.cwd()/'cmri_model_figure_recovery'/'archived_general_pretraining_figures_v3']
archive_dir=next((d for d in archive_candidates if d.is_dir()),None)
if archive_dir:
    archive_files=pd.DataFrame([{'path':str(p)} for p in sorted(archive_dir.glob('*.png'))])
    contact_sheets_from_inventory(archive_files)
else:
    print('No bundled archive folder found on HPC. This is normal if only the .ipynb was uploaded.')


## 4. Inspect the saved *actual* early 192-molecule notebook outputs

Print `evaluation_summary.json` and view `sample_metadata.csv` if present. The original notebook's `H200_EVALUATION_APPROVED=False` is an *old execution gate*, not proof that any old output exists. Do **not** unlock and rerun that notebook merely to look for images.

In [ ]:
for k in ('T4 early n192','H200 early n192'):
    d=EXPECTED[k]
    print('\n===',k,'===',d)
    s=d/'evaluation_summary.json'
    if s.is_file():
        j=json.loads(s.read_text())
        print('Saved summary present; top-level keys:',list(j.keys()))
        print(json.dumps({key:j.get(key) for key in ('notebook','checkpoint','run_mode','n_molecules','sample_seed') if key in j},indent=2,default=str)[:2600])
    else: print('No saved evaluation_summary.json at that expected path.')
    m=d/'sample_metadata.csv'
    if m.is_file(): display(pd.read_csv(m).head(5))
    for name in ('pca.png','pca_by_representation.png','umap.png','embedding_diagnostics.png'):
        print(name, (d/name).exists())

## 5. Inspect the current H200 10K raw evaluation (no inference)

This proves which raw tables exist and displays their real columns, including the previously observed `token_symbol`, `masked_targets`, `accuracy`, `coord_noisy_rmse`, and `coord_pred_rmse`.

In [ ]:
VALID=EVAL/'checkpoint_evaluation/full_10k_batch8'
for name in ['evaluation_summary.json','atom_token_class_accuracy.csv','molecule_diagnostics.csv','objective_batches.csv','cls_view0.npy','cls_view1.npy','projected_view0.npy','projected_view1.npy']:
    p=VALID/name
    print('\n',name,'exists:',p.exists())
    if not p.exists():continue
    if p.suffix=='.json':
        j=json.loads(p.read_text());print(json.dumps(j,indent=2,default=str)[:2600])
    elif p.suffix=='.csv':
        d=pd.read_csv(p,nrows=5);print('COLUMNS:',list(d.columns));display(d.head(3))
    elif p.suffix=='.npy':
        arr=np.load(p,mmap_mode='r');print('shape',arr.shape, 'dtype',arr.dtype)

## 6. Recover / display the validation per-element plot directly from its original CSV

This redraw is explicitly marked as **replotted from original data**. It does not overwrite previous figures or load a checkpoint. It is useful if the old PNG is genuinely missing.

In [ ]:
f=VALID/'atom_token_class_accuracy.csv'
if f.exists():
    el=pd.read_csv(f)
    if {'token_symbol','accuracy','masked_targets'}.issubset(el.columns):
        el=el[el['token_symbol'].astype(str).str.fullmatch('C|N|O|S|F|Cl|Br|I')].copy()
        el['accuracy']=pd.to_numeric(el['accuracy'],errors='coerce')
        el['masked_targets']=pd.to_numeric(el['masked_targets'],errors='coerce')
        # If accuracy is stored as a proportion, use percent. Avoid assuming either convention.
        if el['accuracy'].max()<=1.001: el['accuracy']*=100
        fig,ax=plt.subplots(figsize=(11,4.3))
        colors=plt.cm.YlOrRd(np.linspace(.23,.85,len(el)))
        bars=ax.bar(el['token_symbol'],el['accuracy'],color=colors)
        ax.set(ylabel='Masked token accuracy (%)',title='cmri_model H200 | validation 10K | recovery by element',ylim=(0,110))
        for b,count in zip(bars,el['masked_targets']):
            ax.text(b.get_x()+b.get_width()/2,b.get_height()+1.3,f'{int(count):,}',ha='center',fontsize=8)
        fig.tight_layout();display(fig);plt.close(fig)
    else: print('Missing columns, actual:',list(el.columns))
else: print('Missing original CSV:',f)

## 7. Inspect all previous downstream benchmark results, not just the composite panel

FreeSolv may be labeled `SAMPL`. The benchmark data should be shown from the saved `benchmark_all_runs.csv` without assuming which specific run or label fraction is the original plot.

In [ ]:
p=EVAL/'chemistry_slide_evidence/independent_property_benchmarks/benchmark_all_runs.csv'
if p.exists():
    bench=pd.read_csv(p)
    display(bench.head(12))
    print('COLUMNS:',list(bench.columns))
    if 'benchmark' in bench:
        print('BENCHMARK VALUES:',sorted(bench['benchmark'].dropna().astype(str).unique()))
    if 'model' in bench:
        print('MODEL VALUES:',sorted(bench['model'].dropna().astype(str).unique()))
else: print('Benchmark CSV missing:',p)

## 8. Locate the original *source notebook* code and its declared save targets

Checks both old T4/H200 embedding notebooks and any other nearby notebook code for `savefig`, `OUTPUT_DIR`, and related paths. Prints references; never runs those cells. If no saved early plots exist, this output helps identify the original producer.

In [ ]:
NOTEBOOK_ROOTS=[REPO/'model',REPO/'model/pretrain/notebooks',REPO/'runs/evaluation']
seen=set(); records=[]
for root in NOTEBOOK_ROOTS:
    if not root.exists():continue
    for nb in root.rglob('*.ipynb'):
        if nb in seen:continue
        seen.add(nb)
        try:j=json.loads(nb.read_text())
        except Exception:continue
        for i,c in enumerate(j.get('cells',[])):
            src=''.join(c.get('source',[]))
            hits=[line.strip()[:230] for line in src.splitlines() if any(term in line.lower() for term in ('savefig(', 'output_dir =', 'output_dir  =', 'run_output_dir =', 'pca.png', 'figure_dir ='))]
            for hit in hits[:15]:records.append({'notebook':str(nb),'cell':i,'source_line':hit})
notebook_sources=pd.DataFrame(records)
if not notebook_sources.empty:
    display(notebook_sources.head(100))
    notebook_sources.to_csv(OUTPUT/'notebook_figure_source_map.csv',index=False)
else:print('No notebook output directives found in searched roots.')

## 9. Review / inventory summary

The inventory CSV files are saved into `summary_figures/original_figure_recovery/`. They contain paths and source snippets only, not recomputed evaluation claims. **Only if the original `full_n192_seed17` directory is absent** should you consider re-running that old notebook (after safely repairing paths, provenance checks, and old guards) or reconstructing a purely plotting notebook from saved data.

In [ ]:
print('Report directory:',OUTPUT)
for p in sorted(OUTPUT.glob('*')): print(' ',p.name,p.stat().st_size,'bytes')
print('\nSummary:')
for k,d in EXPECTED.items():
 print(f'  {k:32s}', 'FOUND' if d.is_dir() else 'NOT FOUND',d)
print('\nDo not run the original H200 checkpoint notebook until its checkpoint/run safety conditions and paths have been reviewed.')